Dado 5: Existência de Ciclovias e Ciclofaixas na Região

* Fonte Oficial / Técnica: Referência oficial [ObservaMOB POA](https://prefeitura.poa.br); extração vetorial dedicada via OpenStreetMap (`ox.features_from_place`).
* Justificativa da Extração Dedicada: A extração geral da malha viária (`network_type="bike"`) omite as tags específicas de infraestrutura cicloviária (`cycleway`). Por isso, este notebook realiza uma consulta espacial exclusiva pelas tags `highway=cycleway` e `cycleway=*` em Porto Alegre, gerando uma camada vetorial própria para servir como filtro de exclusão na priorização de novas rotas.
* Etapas do Notebook:
  1. Extração Bruta: Consulta à API do OpenStreetMap pelas feições lineares cicloviárias de Porto Alegre e salvamento em `dados_brutos/bruto_osm_ciclovias_poa.geojson`.
  2. Limpeza e Padronização: Classificação da infraestrutura entre **ciclovia** (via segregada) e **ciclofaixa** (faixa compartilhada na via), seleção de atributos essenciais e exportação para `dados_limpos/limpo_05_ciclovias_existentes.geojson`.

In [3]:
import osmnx as ox

print("Baixando exclusivamente as ciclovias e ciclofaixas de Porto Alegre...")
# Busca todas as geometrias marcadas como ciclovia ou rua com ciclofaixa
tags_ciclovia = {'highway': 'cycleway', 'cycleway': True}
ciclovias_brutas = ox.features_from_place("Porto Alegre, Rio Grande do Sul, Brazil", tags=tags_ciclovia)

# Mantém apenas linhas (trechos de via) e converte colunas para texto para salvar o bruto
ciclovias_brutas = ciclovias_brutas[ciclovias_brutas.geometry.type.isin(['LineString', 'MultiLineString'])].copy()
for col in ciclovias_brutas.columns:
    if col != 'geometry':
        ciclovias_brutas[col] = ciclovias_brutas[col].astype(str)

ciclovias_brutas.to_file("../../data/1_raw/bruto_osm_ciclovias_poa.geojson", driver="GeoJSON")
print(f"Dado bruto salvo! {len(ciclovias_brutas)} trechos cicloviários encontrados em Porto Alegre.")

Baixando exclusivamente as ciclovias e ciclofaixas de Porto Alegre...
Dado bruto salvo! 396 trechos cicloviários encontrados em Porto Alegre.


In [4]:
import geopandas as gpd

ciclovias = gpd.read_file("../../data/1_raw/bruto_osm_ciclovias_poa.geojson")

# Padroniza o tipo de infraestrutura (ciclovia segregada vs ciclofaixa compartilhada)
ciclovias['tipo_infraestrutura'] = ciclovias.apply(
    lambda row: 'ciclovia' if row.get('highway') == 'cycleway' else 'ciclofaixa', axis=1
)
ciclovias['possui_ciclovia'] = True

colunas_uteis = [c for c in ['osmid', 'name', 'tipo_infraestrutura', 'possui_ciclovia', 'geometry'] if c in ciclovias.columns]
ciclovias_limpas = ciclovias[colunas_uteis].copy()

ciclovias_limpas.to_file("../../data/3_done/limpo_05_ciclovias_existentes.geojson", driver="GeoJSON")
print(f"Concluído! {len(ciclovias_limpas)} trechos limpos salvos em 'dados_limpos/'.")

Concluído! 396 trechos limpos salvos em 'dados_limpos/'.
